# Fashion-MNIST: Linear Softmax Classifier

This note studies **multinomial logistic regression**, implemented as one affine layer followed by a softmax function, with no hidden layers and no regularization. The goal is to build a mathematically consistent baseline for later GD and SGD analysis.

**Scope (Part 1).** This notebook covers the model, the loss, its first- and second-order derivatives, and the convexity of the objective in $(W,b)$. The convergence analysis of gradient descent ($L$-smoothness and the $\mathcal O(1/T)$ rate) is in progress and will be added in a later part.

The derivations distinguish derivatives with respect to **logits** from derivatives with respect to the **trainable parameters**.

## 1. Notation and dimensions

Vectors are column vectors in the mathematics. Samples are stored as rows in the data and logit matrices. Lowercase letters denote scalars or vectors, and uppercase letters denote matrices; indices make the distinction explicit.

| Symbol | Meaning | Dimension or range |
| --- | --- | --- |
| $n$ | Number of training samples | $60{,}000$ |
| $d$ | Number of features per image | $28\times28=784$ |
| $K$ | Number of classes | $10$ |
| $i$ | Sample index | $1,\ldots,n$ |
| $r,s$ | Feature indices | $1,\ldots,d$ |
| $j,\ell$ | Class indices | $1,\ldots,K$ |
| $T$ | Raw training images | $\{0,\ldots,255\}^{n\times28\times28}$ |
| $x_i$ | Flattened and normalized image | $[0,1]^d$ |
| $X$ | Matrix with row $i$ equal to $x_i^\top$ | $[0,1]^{n\times d}$ |
| $c_i$ | Class label in mathematical notation | $\{1,\ldots,K\}$ |
| $y_i$ | One-hot target, with $y_{i,j}=\mathbf{1}_{\{j=c_i\}}$ | $\{0,1\}^K$, $\mathbf{1}_K^\top y_i=1$ |
| $Y$ | Matrix with row $i$ equal to $y_i^\top$ | $\{0,1\}^{n\times K}$ |
| $W$ | Trainable weight matrix | $\mathbb{R}^{d\times K}$ |
| $b$ | Trainable bias vector | $\mathbb{R}^K$ |
| $z_i$ | Logit vector for sample $i$ | $\mathbb{R}^K$ |
| $Z$ | Matrix with row $i$ equal to $z_i^\top$ | $\mathbb{R}^{n\times K}$ |
| $p_i$ | Predicted class probabilities for sample $i$ | $(0,1)^K$, $\mathbf{1}_K^\top p_i=1$ |
| $P$ | Matrix with row $i$ equal to $p_i^\top$ | $(0,1)^{n\times K}$ |
| $\mathbf{1}_q$ | Vector of ones | $\mathbb{R}^q$ |

The mathematical class labels are one-based; the Python labels are zero-based, from `0` to `K - 1`. All norms of vectors below are Euclidean. Matrix parameter gradients use the Frobenius inner product, $\langle A,B\rangle_F=\operatorname{tr}(A^\top B)$.

## 2. Load and preprocess Fashion-MNIST

Fashion-MNIST supplies separate training and test sets, with $60{,}000$ and $10{,}000$ images, respectively. The objective below uses only the training set. The test set is reserved for evaluation.

For each image, flatten its $28\times28$ pixels and divide the raw integer intensities by $255$:

$$
x_i=\frac{\operatorname{flatten}(T_i)}{255}\in[0,1]^d.
$$

The code reads `dataset.data` directly. Dataset transforms such as `ToTensor()` are applied when retrieving samples through `dataset[i]`, not when directly accessing `.data`; therefore, the conversion and normalization are explicit here.

**Notebook usage:** Select a Python 3 kernel with `torch` and `torchvision` installed, then run the code cells in order. The first data-loading cell downloads Fashion-MNIST if needed.


In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)


In [2]:
def load_fashion_mnist(root="data", download=False):
    """Read raw uint8 tensors and normalize once; transforms are unnecessary."""
    from torchvision import datasets

    training_data = datasets.FashionMNIST(
        root=root, train=True, download=download
    )
    test_data = datasets.FashionMNIST(
        root=root, train=False, download=download
    )
    X = training_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    labels = training_data.targets.to(torch.long)
    Y = F.one_hot(labels, num_classes=10).to(X.dtype)
    X_test = test_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    test_labels = test_data.targets.to(torch.long)
    return X, Y, labels, X_test, test_labels


In [3]:
# Set download=True for the first run if the dataset is not already present.
X, Y, labels, X_test, test_labels = load_fashion_mnist(
    root="data", download=True
)
n, d = X.shape
K = Y.shape[1]

assert X.shape == (60000, 784)
assert Y.shape == (60000, 10)
assert X_test.shape == (10000, 784)
assert torch.all(Y.sum(dim=1) == 1)


The one-hot matrix $Y$ is convenient for the derivation. An implementation could keep integer labels instead to save memory, but the one-hot form is used here because it matches the formulas.

## 3. Affine model and row-wise softmax

For fixed parameters $(W,b)$, define

$$
z_i=W^\top x_i+b,
\qquad
Z=XW+\mathbf{1}_n b^\top.
$$

In Python, `X @ W + b` broadcasts `b` over samples. The stored shape of `b` remains `(K,)`; broadcasting does not modify the tensor's shape.

The entries of $W$ and $b$ are initialized independently from $U[0,1)$. This is only a starting point for the computations; the derivative formulas and the convexity result do not depend on it.


In [ ]:
def initialize_parameters(d, K, *, dtype=torch.float32, device="cpu"):
    """Entries of W and b sampled from U[0, 1); seed torch beforehand for repeats."""
    W = torch.rand(d, K, dtype=dtype, device=device)
    b = torch.rand(K, dtype=dtype, device=device)
    return W, b


In [5]:
W, b = initialize_parameters(d, K, dtype=X.dtype, device=X.device)
Z = X @ W + b
assert Z.shape == (n, K)


For one sample, define $\sigma:\mathbb{R}^K\to(0,1)^K$ by

$$
\sigma(z_i)_j=p_{i,j}
=\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}.
$$

The corresponding matrix map $\Sigma:\mathbb{R}^{n\times K}\to(0,1)^{n\times K}$ applies $\sigma$ independently to each row, so $P=\Sigma(Z)$. Each row of $P$ sums to one.

**Flattening the dataset does not turn this into one softmax over $nK$ entries.** Every normalization denominator still includes only the $K$ classes of its own sample.

For numerical stability, introduce a separate shifted quantity

$$
m_i=\max_j z_{i,j},\qquad
\widetilde z_i=z_i-m_i\mathbf{1}_K.
$$

Softmax is invariant to this shift. The mathematical model continues to use the original logits $Z=XW+\mathbf{1}_n b^\top$; shifting is an evaluation technique.

## 4. Per-sample loss and scalar objective

With the target $y_i$ fixed, the per-sample negative log-likelihood is

$$
\ell_i:\mathbb{R}^K\to\mathbb{R},\qquad
\ell_i(z_i)
=-\sum_{j=1}^K y_{i,j}\log p_{i,j}
=-\log p_{i,c_i}.
$$

Using $\sum_j y_{i,j}=1$ gives the equivalent expression

$$ 
\begin{aligned}
\ell_i(z_i) &=-\sum_{j=1}^K y_{i,j}\log p_{i,j}\\
&=-\sum_{j=1}^K y_{i,j}\log\left(\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}\right)\\
&=-\left(\sum_{j=1}^K y_{i,j} z_{i,j} - \underbrace{\sum_{j=1}^K y_{i,j}}_{=1} \log\sum_{j=1}^K\exp(z_{i,j}) \right)
\end{aligned}
$$

$$
\boxed{
\ell_i(z_i)=\log\left(\sum_{j=1}^K e^{z_{i,j}}\right)-y_i^\top z_i.
}
$$

For finite logits and $K\ge2$, $0<p_{i,c_i}<1$, so $\ell_i(z_i)>0$ in exact arithmetic. In particular, zero is a lower bound. Floating-point evaluation may round a sufficiently small loss to zero.

Distinguish three objects:

$$
\mathcal{L}(Z)=\begin{bmatrix}\ell_1(z_1)\\\vdots\\\ell_n(z_n)\end{bmatrix}
\in\mathbb{R}^n,
\qquad
\Phi(Z)=\frac1n\sum_{i=1}^n\ell_i(z_i),
$$

$$
\boxed{
f(W,b)=\Phi(XW+\mathbf{1}_n b^\top)
=\frac1n\sum_{i=1}^n f_i(W,b),
\qquad
f_i(W,b)=\ell_i(W^\top x_i+b).
}
$$

$\mathcal{L}$ is vector-valued, whereas $\Phi$ and $f$ are scalar-valued. Training minimizes $f$ over $(W,b)$; the logits are determined by the fixed data and these shared parameters.

### Stable implementation

Computing `torch.log(P)` after forming probabilities can produce `-inf` when a probability underflows to zero. Multiplying this by a zero one-hot entry can then produce `NaN`. Compute log-probabilities directly instead:

$$
\log p_{i,j}=\widetilde z_{i,j}
-\log\left(\sum_{\ell=1}^K e^{\widetilde z_{i,\ell}}\right).
$$


In [6]:
def log_softmax_rows(Z):
    """Stable row-wise log probabilities without taking log(softmax(Z))."""
    shifted = Z - Z.amax(dim=1, keepdim=True)
    return shifted - torch.logsumexp(shifted, dim=1, keepdim=True)


def forward(X, Y, W, b):
    """Y must contain one-hot rows (or valid probability distributions)."""
    Z = X @ W + b
    log_P = log_softmax_rows(Z)
    P = log_P.exp()
    losses = -(Y * log_P).sum(dim=1)
    return P, losses


def objective(X, Y, W, b):
    return forward(X, Y, W, b)[1].mean()


In [7]:
P, losses = forward(X, Y, W, b)
f_value = losses.mean()

assert losses.shape == (n,)
assert f_value.ndim == 0
torch.testing.assert_close(P.sum(dim=1), torch.ones_like(P[:, 0]))
torch.testing.assert_close(f_value, F.cross_entropy(X @ W + b, labels))


## 5. First-order derivatives

### 5.1 Softmax Jacobian for one sample

Temporarily omit the sample index and write $p=\sigma(z)$. For $j,\ell\in\{1,\ldots,K\}$,

$$
\frac{\partial p_j}{\partial z_\ell}
=\begin{cases}
p_j(1-p_j),&j=\ell,\\
-p_jp_\ell,&j\ne\ell.
\end{cases}
=p_j(\delta_{j\ell}-p_\ell).
$$

Thus,

$$
\boxed{J_\sigma(z)=\operatorname{Diag}(p)-pp^\top\in\mathbb{R}^{K\times K}.}
$$

Here $\delta_{j\ell}$ is the Kronecker delta.  

$$
\delta_{j\ell}
=\begin{cases}
1, j = \ell \\
0, j \neq \ell
\end{cases}
$$


Use **Jacobian** notation $J_\sigma$ (or $D\sigma$) for the derivative of this vector-valued function. Its entries are not all positive: the off-diagonal entries are negative for finite logits.

### 5.2 Loss gradient with respect to logits

For one sample, the chain rule gives

$$
\begin{aligned}
\frac{\partial\ell_i}{\partial z_{i,\ell}}
&=-\sum_{j=1}^K y_{i,j}\frac{1}{p_{i,j}}
\frac{\partial p_{i,j}}{\partial z_{i,\ell}}\\
&=-\sum_{j=1}^K y_{i,j}(\delta_{j\ell}-p_{i,\ell})\\
&=-\sum_{j=1}^K y_{i,j}\delta_{j\ell}+\sum_{j=1}^K y_{i,j}p_{i,\ell}\\
&=-y_{i,\ell}+p_{i,\ell}\underbrace{\sum_{j=1}^K y_{i,j}}_{=1}\\
&=p_{i,\ell}-y_{i,\ell}.
\end{aligned}
$$

Therefore,

$$
\boxed{\nabla_{z_i}\ell_i(z_i)=p_i-y_i\in\mathbb{R}^K,}
\qquad
\boxed{\nabla_Z\Phi(Z)=\frac1n(P-Y)\in\mathbb{R}^{n\times K}.}
$$

These gradients can have negative components even though the losses are nonnegative. A nonnegative loss does not imply a nonnegative gradient.

### 5.3 Gradients with respect to weights and bias

Since $z_{i,j}=\sum_{r=1}^d x_{i,r}w_{r,j}+b_j$,

$$
\frac{\partial f}{\partial w_{r,j}}
=\frac1n\sum_{i=1}^n x_{i,r}(p_{i,j}-y_{i,j}),
\qquad
\frac{\partial f}{\partial b_j}
=\frac1n\sum_{i=1}^n(p_{i,j}-y_{i,j}).
$$

In matrix form,

$$
\boxed{\nabla_W f(W,b)=\frac1n X^\top(P-Y)\in\mathbb{R}^{d\times K},}
$$

$$
\boxed{\nabla_b f(W,b)=\frac1n(P-Y)^\top\mathbf{1}_n\in\mathbb{R}^{K}.}
$$


In [8]:
def manual_gradient(X, Y, W, b):
    """Gradient of the MEAN loss; shapes are (d, K) and (K,)."""
    n = X.shape[0]
    P, _ = forward(X, Y, W, b)
    residual = P - Y
    grad_W = X.T @ residual / n
    grad_b = residual.mean(dim=0)
    return grad_W, grad_b


In [9]:
grad_W, grad_b = manual_gradient(X, Y, W, b)
assert grad_W.shape == W.shape  # (784, 10)
assert grad_b.shape == b.shape  # (10,)


**A mistake from my first draft.** In `concepting/gd_sgd_analysis.ipynb` I computed the weight gradient as `(X.T @ (P - Y)).mean(dim=0)`, and it returned shape `(10,)` instead of `(784, 10)`. The reason: the matrix product `X.T @ (P - Y)` already sums over the $n$ samples, so its shape is `(d, K)`. Calling `.mean(dim=0)` afterwards averages over the **feature** dimension $d$, not over samples. The fix is to divide the product by $n$, which is exactly $\frac1n X^\top(P-Y)$ from Section 5.3. The shape asserts above are there to catch this kind of error.

# 6. Second-order derivatives

## Second-order derivatives with respect to the logits $z_i$

From the first-order derivative $ \frac{\partial\ell_i}{\partial z_{i,\ell}} = p_{i,\ell}-y_{i,\ell} $ we differentiate once more. The target $y_{i,\ell}$ is a constant, so only $p_{i,\ell}$ contributes, and we can reuse the softmax Jacobian from Section 5.1.

For the diagonal entries ($j=\ell$):
$$
\frac{\partial^2 \ell_i}{\partial z_{i,\ell}^2} = \frac{\partial \left(p_{i,\ell}-y_{i,\ell}\right)}{\partial z_{i,\ell}} = p_{i,\ell}(1-p_{i,\ell})
$$

For the off-diagonal entries ($j\neq\ell$):
$$
\frac{\partial^2 \ell_i}{\partial z_{i,j}\,\partial z_{i,\ell}} = \frac{\partial \left(p_{i,\ell}-y_{i,\ell}\right)}{\partial z_{i,j}} = -p_{i,j}\,p_{i,\ell}
$$

Both cases can be written as $p_{i,\ell}(\delta_{j\ell}-p_{i,j})$. The Hessian is symmetric, since swapping $j$ and $\ell$ gives the same value. Therefore, define $ C_i $

$$
\boxed{C_i := \nabla_{z}^2\ell_i(z_i)=\operatorname{Diag}(p_i)-p_ip_i^\top\in\mathbb{R}^{K\times K}}
$$

This is the same matrix as the softmax Jacobian $J_\sigma(z_i)$, because $\nabla_z\ell_i = \sigma(z_i) - y_i$.

# 7. Checking convexity

## Definition of Convexity
A function $ f : \mathbb{R}^d \to \mathbb{R} \cup \{+\infty\} $ is convex if for all $ x, y \in \mathbb{R}^d $ and for all $ t \in (0,1),\qquad f(tx+(1-t)y)\leq tf(x) + (1-t)f(y) $

## First-order characterization
If $ f : \mathbb{R}^d \to \mathbb{R} $ is differentiable, then $f$ is convex if and only if, for all $ x, y \in \mathbb{R}^d,\qquad f(x) \geq f(y) + \langle \nabla f(y), x - y \rangle $

## Second-order characterization
If $ f : \mathbb{R}^d \to \mathbb{R} $ is twice differentiable, then $f$ is convex **if and only if** $\nabla^2 f(x)$ is positive semidefinite for all $ x \in \mathbb{R}^d $, that is, every eigenvalue $ \lambda $ of $ \nabla^2 f(x) $ satisfies $ \lambda \geq 0 $. Equivalently, $u^\top \nabla^2 f(x)\, u \ge 0$ for every $x$ and every direction $u$.

We need the "if" direction: showing that the Hessian is positive semidefinite everywhere proves convexity.

## Proof for the per-sample loss
We use the second-order characterization. Instead of computing eigenvalues directly, we check $u^\top C_i u \ge 0$ for every direction $u$. To get this quadratic form naturally, restrict the loss to a line in logit space.

Fix a logit vector $ z_i $ and any direction $ u \in \mathbb{R}^K $, and define the line $ z_i(t) = z_i + tu $ for $ t \in \mathbb{R} $. Let $ h(t) = \ell_i(z_i(t)) $ be the loss along this line. Since $ \frac{d z_i(t)}{dt} = u $, the chain rule gives:
$$
\begin{aligned}
h'(t) &= \underbrace{\nabla_z \ell_i(z_i(t))^\top}_{\in \mathbb{R}^{1 \times K}} \underbrace{\frac{d z_i(t)}{dt}}_{\in \mathbb{R}^{K}} \\
&= \left(p_i(t) - y_i\right)^\top u
\end{aligned}
$$
for the first-order derivative, and
$$
\begin{aligned}
h''(t) = \frac{d h'(t)}{dt} &= \frac{d\left(p_i(t) - y_i \right)^\top u}{dt} \\
&= \underbrace{u^\top}_{\in \mathbb{R}^{1 \times K}} \underbrace{\nabla_{z}^2 \ell_i\left(z_i(t)\right)}_{\in \mathbb{R}^{K \times K}}\underbrace{u}_{\in \mathbb{R}^K} \\
&= u^\top C_i(t)\, u
\end{aligned}
$$
for the second-order derivative, where $C_i(t)$ is $C_i$ evaluated at $z_i(t)$.

We can expand the quadratic form:
$$
\begin{aligned}
u^\top C_i u &= u^\top \left(\operatorname{Diag}(p_i)-p_ip_i^\top\right) u \\
&= u^\top\operatorname{Diag}(p_i)u - u^\top p_i p_i^\top u \\
&= \sum_{j=1}^K p_{i,j} u_j^2 - \left(\sum_{j=1}^K p_{i,j} u_j\right)^2 \\
\end{aligned}
$$

Let $ \bar u = \sum_{j=1}^K p_{i,j} u_j $. Using $\sum_{j=1}^K p_{i,j} = 1$,
$$
\begin{aligned}
\sum_{j=1}^K p_{i,j} u_j^2 - \left(\sum_{j=1}^K p_{i,j} u_j\right)^2 &= \sum_{j=1}^K p_{i,j} u_j^2 - \bar u^2 \\
&= \sum_{j=1}^K p_{i,j} u_j^2 + \bar u^2 - 2 \bar u^2 \\
&= \sum_{j=1}^K p_{i,j} u_j^2 + \bar u^2 \sum_{j=1}^K p_{i,j} - 2 \bar u \left(\sum_{j=1}^K p_{i,j} u_j\right)
\end{aligned}
$$


# 8. Convexity with respect to $W$ and $b$

Section 7 studies one loss $\ell_i$ as a function of its logits. Training, however, minimizes $f(W,b)$, so convexity must hold in the **parameters**. The argument has two ingredients: (i) $C_i\succeq0$ for every logit vector, and (ii) the map $(W,b)\mapsto z_i$ is affine.

## 8.1 Completing the logit-level argument

Continue the expansion at the end of Section 7. With $\bar u=\sum_{j}p_{i,j}u_j$ and $\sum_j p_{i,j}=1$, the last line is a sum of squares:

$$
\begin{aligned}
u^\top C_iu
&=\sum_{j=1}^K p_{i,j}u_j^2-2\bar u\sum_{j=1}^K p_{i,j}u_j+\bar u^2\sum_{j=1}^K p_{i,j}\\
&=\sum_{j=1}^K p_{i,j}\left(u_j^2-2\bar u\,u_j+\bar u^2\right)\\
&=\sum_{j=1}^K p_{i,j}(u_j-\bar u)^2\;\ge\;0.
\end{aligned}
$$

This is the variance of the entries of $u$ under the distribution $p_i$. Hence $C_i\succeq0$ for every $z_i\in\mathbb{R}^K$, so $h''(t)\ge0$ along every line, and each $\ell_i$ is convex in $z_i$.

The inequality is not strict: $C_i\mathbf{1}_K=p_i-p_i(\mathbf{1}_K^\top p_i)=0$. Along $u=\mathbf{1}_K$, every logit increases by the same amount, the softmax output does not change, and the curvature is zero.

## 8.2 From the logits to $(W,b)$

The logits are an **affine** function of the parameters: $z_i = W^\top x_i + b$. So we can reuse the line argument of Section 7.

Take any point $(W,b)$ and any direction $(V,c)$ with $V\in\mathbb{R}^{d\times K}$, $c\in\mathbb{R}^K$. Along the line $(W+tV,\ b+tc)$ the logits are

$$
z_i(t) = (W+tV)^\top x_i + (b+tc) = z_i + t\,u_i,\qquad u_i := V^\top x_i + c.
$$

The direction $u_i$ does not depend on $t$, so a line in parameter space becomes a line in the logit space of every sample. Let $g(t) = f(W+tV,\ b+tc) = \frac1n\sum_{i=1}^n \ell_i(z_i + t u_i)$. Applying Section 7 to each term,

$$
g''(t) = \frac1n\sum_{i=1}^n u_i^\top C_i(t)\, u_i \;\ge\; 0
$$

by 8.1. So $g$ is convex along every line, which means

$$
\boxed{f(W,b)\ \text{is convex in}\ (W,b).}
$$

In short: a convex function composed with an affine map is convex, and an average of convex functions is convex.

## 8.3 Convex, but not strictly convex

Shift the bias by the same constant in every class, $b \mapsto b + \alpha\mathbf 1_K$. Then every logit of every sample moves by $\alpha$, and

$$
\ell_i(z_i+\alpha\mathbf 1_K)=\log\sum_{j=1}^Ke^{z_{i,j}+\alpha}-y_i^\top z_i-\alpha\underbrace{y_i^\top\mathbf 1_K}_{=1}=\ell_i(z_i).
$$

So $f(W, b+\alpha\mathbf 1_K) = f(W,b)$ for every $\alpha$: $f$ is flat along this direction. Consequently $f$ is **not strictly convex** (and not strongly convex), and if a minimizer exists it is not unique.

The next cell checks these statements numerically in float64.

In [ ]:
X_dbl, Y_dbl = X.double(), Y.double()
W_dbl, b_dbl = W.double(), b.double()
generator = torch.Generator().manual_seed(0)

# Section 6 and 8.1: C_i = Diag(p_i) - p_i p_i^T has no negative eigenvalues.
P_dbl, _ = forward(X_dbl[:1000], Y_dbl[:1000], W_dbl, b_dbl)
C = torch.diag_embed(P_dbl) - P_dbl[:, :, None] * P_dbl[:, None, :]
assert torch.linalg.eigvalsh(C).min() >= -1e-12

# Section 8.1: u^T C_i u equals the variance of u under p_i.
u = torch.randn(K, dtype=torch.float64, generator=generator)
quadratic_form = torch.einsum("j,ijl,l->i", u, C, u)
u_bar = P_dbl @ u
variance = (P_dbl * (u - u_bar[:, None]).square()).sum(dim=1)
torch.testing.assert_close(quadratic_form, variance)

# Section 8.2: definition of convexity on a chord between two parameter points.
W2 = torch.randn(d, K, dtype=torch.float64, generator=generator)
b2 = torch.randn(K, dtype=torch.float64, generator=generator)
f1 = objective(X_dbl, Y_dbl, W_dbl, b_dbl)
f2 = objective(X_dbl, Y_dbl, W2, b2)
for t in torch.linspace(0.1, 0.9, 9, dtype=torch.float64):
    f_t = objective(X_dbl, Y_dbl, t * W_dbl + (1 - t) * W2, t * b_dbl + (1 - t) * b2)
    assert f_t <= t * f1 + (1 - t) * f2 + 1e-12

# Section 8.3: shifting b by alpha * 1_K does not change f.
torch.testing.assert_close(
    objective(X_dbl, Y_dbl, W_dbl, b_dbl + 5.0),
    objective(X_dbl, Y_dbl, W_dbl, b_dbl),
)